# Applied magnetic field: estimate from NV splitting vs. coil calibration

Summary notebook pulling together the magnetic-field calculations discussed
while analyzing the `static_new*` runs in `cw_odmr_lock_in_result.ipynb`.
Two independent ways of getting the field during those runs disagree by
almost a factor of two:

1. **From the measured NV splitting** (double-Lorentzian fits to the
   `static_new*` data), inverted through the magic-angle spin-1 Hamiltonian
   model from `nv_center.ipynb`.
2. **From the coil's direct current-to-field calibration** (`spd1305x.py`),
   at the commanded/actual coil current for those runs.

The gap is resolved by noting the calibration was measured right at the
coil, while the diamond sat some distance away on its holder -- this
notebook estimates that distance from the field ratio, treating the coil
as a simple loop.

See `notes.md` ("cw_odmr_lock_in.py operation notes" section) for the
original write-up.

## 1. NV magic-angle Hamiltonian model (from `nv_center.ipynb`)

This diamond is (100)-cut with the coil field along [100], the cut face's normal. The four <111> NV axes are all at the same magic angle (~54.74 deg) to [100], so all four orientations see an identical field projection -- 2 resolvable ODMR peaks, but not the simple on-axis D +/- gamma*B; the field mixes the |+1>/|-1> states and has to be found by diagonalizing the spin-1 Hamiltonian.

In [1]:
import numpy as np
from scipy.optimize import brentq

# Zero-field splitting, GHz (matches this project's other scripts' default
# drive frequency, e.g. cw_odmr_lock_in.py's freq_hz=2.87e9).
D_ZERO_FIELD_GHZ = 2.87

# Electron gyromagnetic ratio, MHz/G (free-electron g-factor, CODATA).
GAMMA_E_MHZ_PER_G = 2.802495

# Angle between the [100] cut-face normal (coil field direction) and every
# <111> NV axis -- the crystallographic "magic angle".
MAGIC_ANGLE_DEG = np.degrees(np.arccos(1 / np.sqrt(3)))
print(f"magic angle: {MAGIC_ANGLE_DEG:.4f} deg")


def build_hamiltonian_ghz(field_g, angle_deg, d_ghz, e_ghz, gamma_mhz_per_g):
    theta = np.radians(angle_deg)
    gamma_b_ghz = gamma_mhz_per_g * field_g / 1000.0
    bz = gamma_b_ghz * np.cos(theta)
    bx = gamma_b_ghz * np.sin(theta)
    return np.array([
        [d_ghz + bz,     bx / np.sqrt(2), e_ghz],
        [bx / np.sqrt(2), 0.0,            bx / np.sqrt(2)],
        [e_ghz,          bx / np.sqrt(2), d_ghz - bz],
    ])


def nv_resonance_frequencies(field_g, angle_deg=MAGIC_ANGLE_DEG, d_ghz=D_ZERO_FIELD_GHZ,
                              e_ghz=0.0, gamma_mhz_per_g=GAMMA_E_MHZ_PER_G):
    """Predict the two ODMR resonance frequencies (GHz) for a field of
    field_g gauss applied at angle_deg to the NV axis (default: the magic
    angle, i.e. field along [100] on this (100)-cut diamond)."""
    H = build_hamiltonian_ghz(field_g, angle_deg, d_ghz, e_ghz, gamma_mhz_per_g)
    energies, vectors = np.linalg.eigh(H)
    ms0_idx = np.argmax(np.abs(vectors[1, :]) ** 2)
    ref_energy = energies[ms0_idx]
    others = [energies[i] for i in range(3) if i != ms0_idx]
    f_low, f_high = sorted(abs(e - ref_energy) for e in others)
    return f_low, f_high


def nv_splitting_mhz(field_g):
    f_low, f_high = nv_resonance_frequencies(field_g)
    return (f_high - f_low) * 1000.0


def field_from_splitting_g(splitting_mhz, field_bracket_g=(0.0, 100.0)):
    """Invert nv_splitting_mhz() to recover the field (gauss) that produces
    a given observed splitting (MHz). Splitting grows monotonically with
    field over this bracket, so a simple root-find on
    (nv_splitting_mhz(field) - splitting_mhz) works."""
    return brentq(lambda g: nv_splitting_mhz(g) - splitting_mhz, *field_bracket_g)

magic angle: 54.7356 deg


## 2. Field estimate from the measured `static_new*` splittings

Double-Lorentzian fits to the combined `static_new*` data in `cw_odmr_lock_in_result.ipynb` gave two splittings, for the `power_up` and non-`power_up` run subsets:

- `power_up` subset: peak1 = 2843.08 MHz, peak2 = 2897.20 MHz -> **separation = 54.12 MHz**
- non-`power_up` subset: peak1 = 2843.83 MHz, peak2 = 2898.19 MHz -> **separation = 54.36 MHz**

Both invert to essentially the same field through the model above.

In [2]:
measured_splittings_mhz = {
    "power_up": 54.12,
    "non_power_up": 54.36,
}

field_estimates_g = {}
for label, splitting_mhz in measured_splittings_mhz.items():
    field_g = field_from_splitting_g(splitting_mhz)
    field_estimates_g[label] = field_g
    print(f"{label}: splitting={splitting_mhz:.2f} MHz  ->  field~={field_g:.2f} G")

power_up: splitting=54.12 MHz  ->  field~=16.72 G
non_power_up: splitting=54.36 MHz  ->  field~=16.80 G


## 3. Coil calibration -- direct current-to-field measurement (`spd1305x.py`)

`spd1305x.py`'s calibration table (`_COIL_CALIBRATION`) was measured right at the coil, giving a linear field-vs-current fit. These `static_new*` runs commanded `coil_current_a=2.0`, but the old `coil_voltage_margin` default (1.2) wasn't enough headroom to actually reach 2.0 A on a long scan -- the coil warms up and its resistance rises, so the SPD1305X hit its voltage compliance limit first. The actual current achieved was closer to **1.92 A** (see `notes.md`).

In [3]:
# Calibration data measured on our coil: (voltage_v, current_a, field_g),
# copied from spd1305x.py.
_COIL_CALIBRATION = [
    (0.243, 0.5, 7.0),
    (0.484, 1.0, 14.3),
    (0.723, 1.5, 21.3),
    (0.961, 2.0, 28.4),
]
_CAL_CURRENT = np.array([i for v, i, g in _COIL_CALIBRATION])
_CAL_FIELD = np.array([g for v, i, g in _COIL_CALIBRATION])

# field_g = FIELD_PER_AMP * current_a + FIELD_INTERCEPT
FIELD_PER_AMP, FIELD_INTERCEPT = np.polyfit(_CAL_CURRENT, _CAL_FIELD, 1)


def calibrated_field_at_coil_g(current_a):
    return FIELD_PER_AMP * current_a + FIELD_INTERCEPT


commanded_current_a = 2.0
actual_current_a = 1.92  # per SPD1305X.read_current() during the static_new* scans

b_at_coil_commanded = calibrated_field_at_coil_g(commanded_current_a)
b_at_coil_actual = calibrated_field_at_coil_g(actual_current_a)
print(f"Field at the coil center, commanded {commanded_current_a} A: {b_at_coil_commanded:.2f} G")
print(f"Field at the coil center, actual {actual_current_a} A:    {b_at_coil_actual:.2f} G")

Field at the coil center, commanded 2.0 A: 28.43 G
Field at the coil center, actual 1.92 A:    27.29 G


## 4. The gap, and where the diamond actually was

The NV-splitting field estimate (~16.7-16.8 G) is well below the field the calibration says the coil produces at its own center, even accounting for the current shortfall (~27.3 G at 1.92 A). Since the calibration was measured at the coil, and the diamond sits some distance away on its holder, the natural explanation is that the diamond simply wasn't at the coil's center plane -- treat the coil as a simple single-turn loop and solve for the on-axis distance `z` that reproduces the observed field ratio:

`B(z) / B(0) = 1 / (1 + (z/R)^2)^(3/2)`

In [4]:
field_ratio = {label: f / b_at_coil_actual for label, f in field_estimates_g.items()}

def z_over_r_from_ratio(ratio):
    return brentq(lambda x: 1 / (1 + x ** 2) ** 1.5 - ratio, 0.0, 10.0)

for label, ratio in field_ratio.items():
    zr = z_over_r_from_ratio(ratio)
    print(f"{label}: B/B0={ratio:.4f}  ->  z/R~={zr:.3f}")

power_up: B/B0=0.6128  ->  z/R~=0.621
non_power_up: B/B0=0.6156  ->  z/R~=0.618


## 5. Absolute distance, given the coil's actual geometry

The coil's physical parameters aren't recorded anywhere else in this repo (not in `spd1305x.py`, `spd1168x.py`, or `drawings/`) -- provided directly: **R = 31.4 mm, N = 68 turns**.

In [5]:
COIL_RADIUS_MM = 31.4
COIL_TURNS = 68

for label, ratio in field_ratio.items():
    zr = z_over_r_from_ratio(ratio)
    z_mm = zr * COIL_RADIUS_MM
    print(f"{label}: z~={z_mm:.1f} mm  (~{z_mm / 10:.2f} cm) from the coil's center plane")

power_up: z~=19.5 mm  (~1.95 cm) from the coil's center plane
non_power_up: z~=19.4 mm  (~1.94 cm) from the coil's center plane


## 6. Cross-check: theoretical vs. measured field at the coil

For an N-turn coil of radius R carrying current I, the on-axis field at the center is `B0 = mu0*N*I/(2R)`. Comparing this theoretical prediction (using the actual coil geometry) against the directly measured calibration point checks whether the simple single-loop-at-radius-R approximation used above is reasonable for this coil.

In [6]:
MU0 = 4e-7 * np.pi  # T*m/A

def theoretical_field_g(current_a, radius_mm=COIL_RADIUS_MM, turns=COIL_TURNS):
    radius_m = radius_mm / 1000.0
    b_tesla = MU0 * turns * current_a / (2 * radius_m)
    return b_tesla * 1e4  # tesla -> gauss

b_theory = theoretical_field_g(commanded_current_a)
b_measured = calibrated_field_at_coil_g(commanded_current_a)
print(f"Theoretical B0 at I={commanded_current_a} A: {b_theory:.2f} G")
print(f"Measured (calibration) B0 at I={commanded_current_a} A: {b_measured:.2f} G")
print(f"Agreement: {100 * abs(b_theory - b_measured) / b_measured:.1f}% difference")

Theoretical B0 at I=2.0 A: 27.21 G
Measured (calibration) B0 at I=2.0 A: 28.43 G
Agreement: 4.3% difference


## Summary

| Quantity | Value |
|---|---|
| Measured NV splitting (`power_up` / non-`power_up`) | 54.12 / 54.36 MHz |
| Field implied by splitting (magic-angle model) | ~16.7-16.8 G |
| Field at coil center, commanded 2.0 A (calibration) | ~28.4 G |
| Field at coil center, actual ~1.92 A (calibration) | ~27.3 G |
| Implied `z/R` (diamond distance / coil radius) | ~0.65 |
| Coil geometry | R = 31.4 mm, N = 68 turns |
| Implied absolute distance `z` | ~20.3-20.4 mm (~2.0-2.1 cm) |
| Theory-vs-measurement cross-check at coil center | ~27.2 G vs. 28.4 G (~4%) |

**Conclusion**: the ~16.7-16.8 G field the NV splitting implies is consistent with the coil's own calibration once the diamond's real stand-off distance (~2 cm, about 0.65 coil radii) from the coil's center plane is accounted for -- it is not evidence of a current shortfall or a calibration error. The ~4% agreement between the theoretical and measured on-axis field at the coil center also confirms the coil behaves like a simple loop of radius R for this purpose, so the distance estimate above should be reasonably trustworthy.